# delta 方法：用 nlcom 计算样本均值非线性变换的标准误

> 对应代码：`MathStatsCode/code_in_notes/Chap.8/nlcom_exp_mean.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**实验目的**：设 $x_i\sim\mathcal{N}(\mu,\sigma^2)$，由 delta 方法可知 $\sqrt{N}\left[e^{\bar{x}}-e^{\mu}\right]\overset{a}{\sim}\mathcal{N}(0,e^{2\mu}\sigma^2)$。本例用 `reg` 得到样本均值及其标准误，再用 `nlcom` 自动完成 delta 方法，得到 $e^{\bar{x}}$ 的标准误与置信区间。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致。


In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


**产生与例中相同的模拟数据**

生成 $N=1000$ 个来自 $\mathcal{N}(1,2)$ 的随机数：`rnormal()` 产生标准正态随机数，乘以 $\sqrt{2}$ 再加 1 即得到均值为 1、方差为 2 的正态样本。`set seed 1900` 固定随机种子，保证结果可复现。


In [2]:
%%stata
clear all
set obs 1000
set seed 1900
gen x = 1+rnormal()*sqrt(2)



. clear all

. set obs 1000
Number of observations (_N) was 0, now 1,000.

. set seed 1900

. gen x = 1+rnormal()*sqrt(2)

. 


**使用回归计算均值**

只包含常数项的回归 `reg x`，其系数 `_b[_cons]` 就是样本均值 $\bar{x}$，同时回归还会报告其标准误（即 $s/\sqrt{N}$）。用回归而不是 `su` 的好处是：回归结果进入了 Stata 的估计结果系统（`e()`），从而可以接着使用 `nlcom` 等后估计命令。


In [3]:
%%stata
reg x



      Source |       SS           df       MS      Number of obs   =     1,000
-------------+----------------------------------   F(0, 999)       =      0.00
       Model |           0         0           .   Prob > F        =         .
    Residual |  2025.08615       999  2.02711326   R-squared       =    0.0000
-------------+----------------------------------   Adj R-squared   =    0.0000
       Total |  2025.08615       999  2.02711326   Root MSE        =    1.4238



------------------------------------------------------------------------------
           x | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
       _cons |   .9629104   .0450235    21.39   0.000      .874559    1.051262
------------------------------------------------------------------------------


**delta method计算标准误**

`nlcom exp(_b[_cons])` 计算估计量的非线性变换 $e^{\bar{x}}$，并使用 **delta 方法**自动给出其标准误：由 $\frac{\partial e^{\mu}}{\partial\mu}=e^{\mu}$，有
$$\se\left(e^{\bar{x}}\right)=e^{\bar{x}}\cdot\se\left(\bar{x}\right),$$
同时 `nlcom` 还会报告基于正态近似的 $z$ 值、$p$ 值与 95% 置信区间。本例中真值 $\mu=1$，从而 $e^{\mu}=e\approx2.718$，可以对照检验 delta 方法给出的标准误是否接近理论值 $e^{\mu}\sigma/\sqrt{N}=e\times\sqrt{2}/\sqrt{1000}\approx0.1215$。


In [4]:
%%stata
nlcom exp(_b[_cons])



       _nl_1: exp(_b[_cons])

------------------------------------------------------------------------------
           x | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
       _nl_1 |   2.619309   .1179304    22.21   0.000     2.388169    2.850448
------------------------------------------------------------------------------
